# Lab 7 - Nowoczesna Hurtownia Danych i Lakehouse w Apache Spark (PySpark & Delta Lake)

Celem niniejszego laboratorium jest poznanie zasad projektowania i budowy nowoczesnych hurtowni danych z wykorzystaniem **Apache Spark (PySpark)** oraz technologii **Delta Lake**. 

W trakcie zajęć połączymy dwa kluczowe podejścia współczesnej inżynierii danych:
1. **Wzorzec potoku przetwarzania danych (Data Pipeline): Architektura Medalionowa (Bronze $\rightarrow$ Silver $\rightarrow$ Gold)**.
2. **Model logiczny organizacji danych analitycznych (Data Modeling): Schemat Gwiazdy Ralpha Kimballa (Star Schema - Fakty i Wymiary w warstwie Gold)**.

## 1. Wprowadzenie Teoretyczne: Trzy wymiary projektowania danych

W praktyce inżynierii danych często spotyka się pozornie sprzeczne pojęcia. Aby uniknąć nieporozumień, rozróżnijmy trzy **niezależne wymiary**:

```
+----------------------------------------------------------------------------------------+
| 1. INFRASTRUKTURA / SILNIK (Gdzie fizycznie leżą dane i co nimi zarządza?)             |
|    - Klasyczne DWH (bazy relacyjne SQL, np. PostgreSQL, Snowflake)                     |
|    - Data Lake (surowe pliki na storage: CSV, Parquet - brak ACID, brak schema enforce)|
|    - Data Lakehouse (Parquet + transakcyjna warstwa metadanych ACID, np. Delta Lake)   |
+----------------------------------------------------------------------------------------+
                                           |
                                           v
+----------------------------------------------------------------------------------------+
| 2. WZORZEC PRZEPŁYWU DANYCH (Jak uszlachetniamy dane w procesie inżynierskim?)         |
|    Architektura Medalionowa:                                                           |
|    [BRONZE: Surowe dane 1:1] --> [SILVER: Oczyszczone i zwalidowane] --> [GOLD: Biznes]|
+----------------------------------------------------------------------------------------+
                                           |
                                           v
+---------------------------------------------------------------------------------------+
| 3. MODEL LOGICZNY W WARSTWIE GOLD (Jak organizujemy tabele pod analitykę BI/OLAP?)     |
|    - Schemat Gwiazdy (Star Schema wg Kimballa):                                       |
|      * Tabele Wymiarów (Dimensions): kontekst biznesowy (Kto? Co? Kiedy?)             |
|      * Tabela Faktów (Facts): zdarzenia biznesowe i metryki liczbowe + klucze obce   |
+---------------------------------------------------------------------------------------+
```

### Dlaczego Schemat Gwiazdy w warstwie Gold?
* **Wymiary (Dimensions):** przechowują atrybuty opisowe (np. `dim_user`, `dim_recipe`, `dim_date`). W hurtowni stosujemy **klucze sztuczne (Surrogate Keys)** – liczby całkowite uniezależniające hurtownię od zmian w systemach źródłowych.
* **Fakty (Facts):** przechowują miary liczbowe i zdarzenia biznesowe (np. `fact_reviews` z oceną `stars`, `reply_count`, `thumbs_up`).
* **Wydajność OLAP:** zapytania analityczne łączą centralną tabelę faktów z wymiarami za pomocą prostych złączeń (tzw. *Star Join*), co pozwala silnikowi Spark na agresywne optymalizacje.

---

**Baza OLAP** (*Online Analytical Processing*) to baza danych zoptymalizowana pod kątem **szybkiego wykonywania złożonych zapytań analitycznych i agregacji na ogromnych zbiorach danych** (od milionów do bilionów wierszy).

W przeciwieństwie do baz transakcyjnych (**OLTP**, np. PostgreSQL, MySQL), które służą do błyskawicznego zapisywania i odczytywania pojedynczych rekordów (np. obsługa koszyka w sklepie internetowym), systemy OLAP służą do odpowiadania na pytania strategiczne, np. *„Jaka była łączna sprzedaż w podziale na kraje i kategorie produktów w ciągu ostatnich 3 lat?”*.

---

### Kluczowe różnice techniczne: OLAP vs OLTP

| Cecha | OLTP (*Online Transaction Processing*) | OLAP (*Online Analytical Processing*) |
| --- | --- | --- |
| **Główny cel** | Transakcje biznesowe w czasie rzeczywistym | Analityka, raportowanie, BI, eksploracja danych |
| **Profil operacji** | Mnóstwo szybkich `INSERT`, `UPDATE`, `DELETE` | Masowy zapis wsadowy (`COPY`/`BULK INSERT`), głównie ciężkie `SELECT` |
| **Układ danych** | **Wierszowy** (*Row-oriented*) | **Kolumnowy** (*Column-oriented*) |
| **Zakres zapytania** | Zwraca kilka wierszy, ale wszystkie kolumny | Czyta miliony wierszy, ale tylko kilka kolumn |
| **Wartości agregowane** | Rzadko (obciążają system transakcyjny) | Podstawowa funkcja (`SUM`, `AVG`, `COUNT`, `GROUP BY`) |
| **Stopień normalizacji** | Wysoki (3NF, unikanie redundancji) | Denormalizacja (schemat gwiazdy, płatki śniegu, jedna płaska tabela) |
| **Przykłady technologii** | PostgreSQL, MySQL, SQL Server, Oracle | ClickHouse, Snowflake, DuckDB, Google BigQuery, Trino, Druid |

---

### Dlaczego bazy OLAP są tak szybkie?

Architektura systemów OLAP opiera się na trzech filarach:

1. **Przechowywanie kolumnowe (Columnar Storage):**
* Jeśli tabela ma 200 kolumn, a Twoje zapytanie agreguje tylko `kwota` i grupuje po `kraj`, silnik OLAP czyta z dysku wyłącznie dane z tych dwóch kolumn. Reszta 198 kolumn w ogóle nie jest ładowana do pamięci RAM.


2. **Ekstremalna kompresja:**
* Przechowywanie danych tego samego typu obok siebie (np. same liczby całkowite lub same kody ISO krajów) pozwala na użycie wydajnych algorytmów kompresji (Run-Length Encoding, Delta Encoding, ZSTD). Zbiory danych zmniejszają się często 5–10-krotnie.


3. **Wektorowe przetwarzanie zapytań (Vectorized Execution):**
* Zamiast przetwarzać dane wiersz po wierszu (jak w klasycznym RDBMS), silniki OLAP przetwarzają dane w paczkach (wektorach), w pełni wykorzystując instrukcje **SIMD** (*Single Instruction, Multiple Data*) nowoczesnych procesorów.



---

### Gdzie znajduje się OLAP w architekturze danych?

W nowoczesnym stosie Big Data / Lakehouse bazy OLAP występują w dwóch głównych rolach:

* **Hurtownia danych / Lakehouse (np. Snowflake, BigQuery, Databricks SQL):** Stanowią warstwę docelową (*Gold layer*), w której dane z całego przedsiębiorstwa są ujednolicone i dostępne dla analityków.
* **Real-time / Interactive OLAP (np. ClickHouse, Apache Pinot, Apache Druid):** Skrajnie zoptymalizowane silniki służące do serwowania analityki z opóźnieniem rzędu milisekund bezpośrednio do aplikacji klienckich, pulpitów BI czy wykrywania anomalii w czasie rzeczywistym.

## 2. Przygotowanie środowiska Spark i Delta Lake

Aby umożliwić Sparkowi obsługę formatu **Delta Lake** oraz zaawansowanych funkcji tabelarycznych Spark SQL, konfigurujemy sesję `SparkSession` ze wsparciem dla rozszerzeń Delty oraz katalogu Spark Catalog.

In [ ]:
# instalacja pakietu delta-spark
# aktualnie w "moim" projekcie jest to wersja 4.2.0
!pip install delta-spark

**Dodatkowo** w systemie Windows wymagane jest zainstalowanie `winutils.exe` oraz konfiguracja hadoop w postacji zmiennej `HADOOP_HOME` (patrz konfiguracja w lab 06).

In [ ]:
import os
import sys

# Konfiguracja interpretera Pythona dla sterownika i executorów Sparka
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_unixtime, to_date, length, trim, when, year, month,
    quarter, dayofmonth, dayofweek, date_format, row_number, monotonically_increasing_id, round, avg, count
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# Inicjalizacja SparkSession z obsługą Delta Lake
spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("DataWarehouse_Lakehouse")
    .config("spark.jars.packages", "io.delta:delta-spark_2.13:4.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.warehouse.dir", "../data/lakehouse/warehouse")
    .getOrCreate()
)

# Ograniczenie nadmiarowych komunikatów logowania
spark.sparkContext.setLogLevel("ERROR")
print(f"Sesja Spark uruchomiona pomyślnie! Wersja Sparka: {spark.version}")

In [ ]:
spark

## 3. Krok 1: Warstwa BRONZE (Raw Ingestion)

**Cel warstwy Bronze:** 
Zapisanie surowych danych w niezmienionej postaci do formatu Delta Lake. Warstwa ta stanowi niezmienną historię danych źródłowych (*immutable audit log*), umożliwiającą ich ponowne przetworzenie w razie zmiany logiki biznesowej.

Wczytamy zbiór recenzji kulinarnych `Recipe Reviews and User Feedback Dataset.csv` znajdujący się w katalogu `../data/`.

In [3]:
csv_path = "../data/Recipe Reviews and User Feedback Dataset.csv"

# Wczytanie surowego pliku CSV (zwróć uwagę na obsługę tekstu wieloliniowego)
df_raw = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option('escape', '"')
    .csv(csv_path)
)

print(f"Liczba wierszy w surowym zbiorze: {df_raw.count()}")
print(f"Kolumny surowe: {df_raw.columns}")
df_raw.select("recipe_name", "user_name", "stars", "created_at").show(5, truncate=False)

Liczba wierszy w surowym zbiorze: 18182
Kolumny surowe: ['_c0', 'recipe_number', 'recipe_code', 'recipe_name', 'comment_id', 'user_id', 'user_name', 'user_reputation', 'created_at', 'reply_count', 'thumbs_up', 'thumbs_down', 'stars', 'best_score', 'text']
+------------------+----------+-----+----------+
|recipe_name       |user_name |stars|created_at|
+------------------+----------+-----+----------+
|Creamy White Chili|Jeri326   |5    |1665619889|
|Creamy White Chili|Mark467   |5    |1665277687|
|Creamy White Chili|Barbara566|5    |1664404557|
|Creamy White Chili|jeansch123|0    |1661787808|
|Creamy White Chili|camper77  |0    |1664913823|
+------------------+----------+-----+----------+
only showing top 5 rows


In [4]:
# Zapis surowych danych do warstwy Bronze w formacie Delta
bronze_path = "../data/lakehouse/bronze/recipe_reviews"

(
    df_raw.write
    .format("delta")
    .mode("overwrite")
    .save(bronze_path)
)

print("Warstwa BRONZE została pomyślnie zapisana pod adresem:", bronze_path)

Warstwa BRONZE została pomyślnie zapisana pod adresem: ../data/lakehouse/bronze/recipe_reviews


## 4. Krok 2: Warstwa SILVER - warstwa oczyszczona (Cleaned, Typed & Enriched)

**Cel warstwy Silver:**
Oczyszczenie danych, standaryzacja schematu, konwersja typów danych, usunięcie duplikatów i wzbogacenie rekordów. Warstwa Silver stanowi tzw. *Single Source of Truth* dla całego przedsiębiorstwa.

W naszym przypadku wykonujemy:
1. Usunięcie technicznej kolumny indeksu `_c0` (lub pustej nazwy `""`).
2. Filtrację rekordów z brakującymi kluczami (`comment_id`, `recipe_code`, `user_id`).
3. Konwersję timestampu unixowego (`created_at`) na typ `Timestamp` oraz datę kalendarzową (`review_date`).
4. Rzutowanie kolumn liczbowych (`recipe_code`, `user_reputation`, `stars`, `reply_count`, `thumbs_up`, `thumbs_down`, `best_score`) na typ całkowity `integer`.
5. Oczyszczenie pól tekstowych funkcją `trim`.
6. Deduplikację na podstawie unikalnego identyfikatora recenzji (`comment_id`).

In [5]:
# Odczyt danych z warstwy Bronze
df_bronze = spark.read.format("delta").load(bronze_path)

# Transformacja do warstwy Silver
df_silver = (
    df_bronze
    .drop("_c0", "")
    .filter(
        col("comment_id").isNotNull() & 
        col("recipe_code").isNotNull() & 
        col("user_id").isNotNull()
    )
    .withColumn("recipe_code", col("recipe_code").cast("integer"))
    .withColumn("user_reputation", col("user_reputation").cast("integer"))
    .withColumn("reply_count", col("reply_count").cast("integer"))
    .withColumn("thumbs_up", col("thumbs_up").cast("integer"))
    .withColumn("thumbs_down", col("thumbs_down").cast("integer"))
    .withColumn("stars", col("stars").cast("integer"))
    .withColumn("best_score", col("best_score").cast("integer"))
    .withColumn("review_timestamp", from_unixtime(col("created_at").cast("long")).cast("timestamp"))
    .withColumn("review_date", to_date(col("review_timestamp")))
    .withColumn("recipe_name", trim(col("recipe_name")))
    .withColumn("user_name", trim(col("user_name")))
    .withColumn("text", trim(col("text")))
    .dropDuplicates(["comment_id"])
)

print(f"Liczba rekordów w warstwie Silver: {df_silver.count()}")
df_silver.printSchema()

Liczba rekordów w warstwie Silver: 18182
root
 |-- recipe_number: string (nullable = true)
 |-- recipe_code: integer (nullable = true)
 |-- recipe_name: string (nullable = true)
 |-- comment_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- user_name: string (nullable = true)
 |-- user_reputation: integer (nullable = true)
 |-- created_at: string (nullable = true)
 |-- reply_count: integer (nullable = true)
 |-- thumbs_up: integer (nullable = true)
 |-- thumbs_down: integer (nullable = true)
 |-- stars: integer (nullable = true)
 |-- best_score: integer (nullable = true)
 |-- text: string (nullable = true)
 |-- review_timestamp: timestamp (nullable = true)
 |-- review_date: date (nullable = true)



In [6]:
# Zapis oczyszczonych danych do warstwy Silver
silver_path = "../data/lakehouse/silver/recipe_reviews"

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .save(silver_path)
)

print("Warstwa SILVER została pomyślnie zapisana pod adresem:", silver_path)

Warstwa SILVER została pomyślnie zapisana pod adresem: ../data/lakehouse/silver/recipe_reviews


## 5. Krok 3: Warstwa GOLD – Modelowanie Schematu Gwiazdy (Kimball)

W warstwie Gold przekształcamy znormalizowane dane ze szczebla Silver w klasyczny **Schemat Gwiazdy**:
* **Wymiar Użytkowników (`dim_user`)**
* **Wymiar Przepisów (`dim_recipe`)**
* **Wymiar Kalendarzowy / Czasu (`dim_date`)**
* **Centralna Tabela Faktów Recenzji (`fact_reviews`)**

```
               +----------------------+
               |       dim_user       |
               +----------------------+
               | PK  user_key         |
               |     user_id (biz)    |
               |     user_name        |
               |     user_reputation  |
               +----------+-----------+
                          |
                          | 1:N
                          v
+------------------+   +----------------------+   +----------------------+
|    dim_recipe    |   |     fact_reviews     |   |       dim_date       |
+------------------+   +----------------------+   +----------------------+
| PK  recipe_key   |<--| PK  review_id        |-->| PK  date_key         |
|     recipe_code  |1:N| FK  user_key         |N:1|     full_date        |
|     recipe_name  |   | FK  recipe_key       |   |     year             |
|     recipe_number|   | FK  date_key         |   |     quarter          |
+------------------+   |     stars            |   |     month            |
                       |     reply_count      |   |     month_name       |
                       |     thumbs_up        |   |     day              |
                       |     thumbs_down      |   |     day_of_week      |
                       |     best_score       |   |     day_name         |
                       |     review_char_count|   |     is_weekend       |
                       +----------------------+   +----------------------+
```

### 5.1. Wymiar Użytkowników (`dim_user`)

> **Ważna uwaga inżynierska:**
> Jeden użytkownik mógł dodać wiele recenzji w różnym czasie (i mieć wówczas inną reputację). W wymiarze użytkownika chcemy mieć dokładnie **jeden unikalny wiersz per `user_id`** z najświeższą wartością reputacji.
> 
> Używamy funkcji okna `row_number().over(...)`, sortując malejąco po dacie recenzji. Użycie `dense_rank()` mogłoby w przypadku remisów czasowych zwrócić dwa wiersze dla tego samego użytkownika, powodując tzw. *fan-out join* (niepożądane powielenie wierszy w tabeli faktów)!

In [7]:
user_window = Window.partitionBy("user_id").orderBy(col("review_timestamp").desc(), col("comment_id"))

dim_user = (
    df_silver
    .withColumn("rn", row_number().over(user_window))
    .filter(col("rn") == 1)
    .select("user_id", "user_name", "user_reputation")
    .withColumn("user_key", monotonically_increasing_id())
)

user_gold_path = "../data/lakehouse/gold/dim_user"
dim_user.write.format("delta").mode("overwrite").save(user_gold_path)

print(f"Liczba unikalnych użytkowników w wymiarze dim_user: {dim_user.count()}")
dim_user.show(5)

Liczba unikalnych użytkowników w wymiarze dim_user: 13812
+--------------+-----------+---------------+--------+
|       user_id|  user_name|user_reputation|user_key|
+--------------+-----------+---------------+--------+
|u_05PZUpOV27Pv|Meredith404|              1|       0|
|u_09Pspx0F3ZKy|  Linda9044|              1|       1|
|u_0BYS3gNJ4rI0|      Sinah|              1|       2|
|u_0GfixeKJgmAL|   Linda408|              1|       3|
|u_0HraB0BMR3qu|    home524|              1|       4|
+--------------+-----------+---------------+--------+
only showing top 5 rows


### 5.2. Wymiar Przepisów (`dim_recipe`)

Wymiar zawiera unikalne przepisy kulinarne z ich kodem, nazwą oraz numerem porządkowym wraz z wygenerowanym sztucznym kluczem `recipe_key`.

In [8]:
dim_recipe = (
    df_silver
    .select("recipe_code", "recipe_name", "recipe_number")
    .dropDuplicates(["recipe_code"])
    .withColumn("recipe_key", monotonically_increasing_id())
)

recipe_gold_path = "../data/lakehouse/gold/dim_recipe"
dim_recipe.write.format("delta").mode("overwrite").save(recipe_gold_path)

print(f"Liczba unikalnych przepisów w dim_recipe: {dim_recipe.count()}")
dim_recipe.show(5)

Liczba unikalnych przepisów w dim_recipe: 100
+-----------+--------------------+-------------+----------+
|recipe_code|         recipe_name|recipe_number|recipe_key|
+-----------+--------------------+-------------+----------+
|        386|       Hot Milk Cake|          026|         0|
|        414|Lemon Blueberry B...|          059|         1|
|        957|Sandy’s Chocolate...|          024|         2|
|       1063| Frosted Banana Bars|          038|         3|
|       1081|     Baked Spaghetti|          022|         4|
+-----------+--------------------+-------------+----------+
only showing top 5 rows


### 5.3. Wymiar Kalendarzowy (`dim_date`)

W hurtowniach danych wymiar daty jest kluczowy – eliminuje konieczność wielokrotnego, kosztownego parsowania dat podczas zapytań analitycznych. Jako klucz główny (`date_key`) tradycyjnie stosuje się liczbę całkowitą w formacie `YYYYMMDD` (np. `20221013`).

In [9]:
dim_date = (
    df_silver
    .select("review_date")
    .filter(col("review_date").isNotNull())
    .dropDuplicates(["review_date"])
    .withColumn("date_key", date_format(col("review_date"), "yyyyMMdd").cast("integer"))
    .withColumn("full_date", col("review_date"))
    .withColumn("year", year(col("review_date")))
    .withColumn("quarter", quarter(col("review_date")))
    .withColumn("month", month(col("review_date")))
    .withColumn("month_name", date_format(col("review_date"), "MMMM"))
    .withColumn("day", dayofmonth(col("review_date")))
    .withColumn("day_of_week", dayofweek(col("review_date")))
    .withColumn("day_name", date_format(col("review_date"), "EEEE"))
    .withColumn("is_weekend", when(col("day_of_week").isin(1, 7), True).otherwise(False))
    .select("date_key", "full_date", "year", "quarter", "month", "month_name", "day", "day_of_week", "day_name", "is_weekend")
)

date_gold_path = "../data/lakehouse/gold/dim_date"
dim_date.write.format("delta").mode("overwrite").save(date_gold_path)

print(f"Liczba unikalnych dat w dim_date: {dim_date.count()}")
dim_date.show(5)

Liczba unikalnych dat w dim_date: 443
+--------+----------+----+-------+-----+----------+---+-----------+--------+----------+
|date_key| full_date|year|quarter|month|month_name|day|day_of_week|day_name|is_weekend|
+--------+----------+----+-------+-----+----------+---+-----------+--------+----------+
|20211011|2021-10-11|2021|      4|   10|   October| 11|          2|  Monday|     false|
|20210827|2021-08-27|2021|      3|    8|    August| 27|          6|  Friday|     false|
|20220328|2022-03-28|2022|      1|    3|     March| 28|          2|  Monday|     false|
|20210622|2021-06-22|2021|      2|    6|      June| 22|          3| Tuesday|     false|
|20211113|2021-11-13|2021|      4|   11|  November| 13|          7|Saturday|      true|
+--------+----------+----+-------+-----+----------+---+-----------+--------+----------+
only showing top 5 rows


### 5.4. Centralna Tabela Faktów (`fact_reviews`)

Tabela faktów łączy zdarzenia biznesowe (wystawienie recenzji) z wymiarami za pomocą kluczy obcych (`user_key`, `recipe_key`, `date_key`). Dodatkowo tworzymy miarę addytywną `review_char_count` określającą długość komentarza w znakach.

In [10]:
fact_reviews = (
    df_silver
    .join(dim_user.select("user_id", "user_key"), on="user_id", how="left")
    .join(dim_recipe.select("recipe_code", "recipe_key"), on="recipe_code", how="left")
    .withColumn("date_key", date_format(col("review_date"), "yyyyMMdd").cast("integer"))
    .select(
        col("comment_id").alias("review_id"),
        "user_key",
        "recipe_key",
        "date_key",
        "stars",
        "reply_count",
        "thumbs_up",
        "thumbs_down",
        "best_score",
        length(col("text")).alias("review_char_count")
    )
)

fact_gold_path = "../data/lakehouse/gold/fact_reviews"

(
    fact_reviews.write
    .format("delta")
    .mode("overwrite")
    .save(fact_gold_path)
)

print(f"Liczba faktów w tabeli fact_reviews: {fact_reviews.count()}")
fact_reviews.show(5)

Liczba faktów w tabeli fact_reviews: 18182
+--------------------+--------+----------+--------+-----+-----------+---------+-----------+----------+-----------------+
|           review_id|user_key|recipe_key|date_key|stars|reply_count|thumbs_up|thumbs_down|best_score|review_char_count|
+--------------------+--------+----------+--------+-----+-----------+---------+-----------+----------+-----------------+
|sp_aUSaElGf_10027...|     488|        57|20210603|    0|          0|        0|          0|       100|               84|
|sp_aUSaElGf_10027...|    1131|        57|20210603|    5|          0|        0|          0|       100|              133|
|sp_aUSaElGf_10027...|    2274|        57|20210603|    5|          0|        0|          0|       100|               33|
|sp_aUSaElGf_10027...|    2375|        57|20210603|    5|          0|        0|          0|       100|              138|
|sp_aUSaElGf_10027...|    3024|        57|20210603|    5|          0|        0|          0|       100|        

## 6. Krok 4: Zaawansowana Analityka OLAP w Spark SQL

Po zbudowaniu Schematu Gwiazdy rejestrujemy tabele w katalogu tymczasowym Spark SQL, aby analitycy biznesowi mogli zadawać zaawansowane zapytania SQL.

In [11]:
# Rejestracja widoków tymczasowych w Spark SQL
dim_user.createOrReplaceTempView("dim_user")
dim_recipe.createOrReplaceTempView("dim_recipe")
dim_date.createOrReplaceTempView("dim_date")
fact_reviews.createOrReplaceTempView("fact_reviews")

print("Tabele zostały pomyślnie zarejestrowane w Spark SQL!")

Tabele zostały pomyślnie zarejestrowane w Spark SQL!


### 6.1. Złączenie Schematu Gwiazdy (Star Join)
Sprawdzamy średnią ocenę oraz liczbę recenzji dla poszczególnych przepisów w podziale na lata.

In [12]:
query_star_join = """
SELECT 
    d.year,
    r.recipe_name,
    COUNT(f.review_id) AS total_reviews,
    ROUND(AVG(f.stars), 2) AS avg_stars,
    SUM(f.thumbs_up) AS total_likes
FROM fact_reviews f
JOIN dim_recipe r ON f.recipe_key = r.recipe_key
JOIN dim_date d ON f.date_key = d.date_key
GROUP BY d.year, r.recipe_name
HAVING COUNT(f.review_id) >= 10
ORDER BY d.year DESC, avg_stars DESC
LIMIT 10
"""

spark.sql(query_star_join).show(truncate=False)

+----+----------------------------+-------------+---------+-----------+
|year|recipe_name                 |total_reviews|avg_stars|total_likes|
+----+----------------------------+-------------+---------+-----------+
|2022|Slow-Cooker Lasagna         |12           |4.83     |141        |
|2022|Chunky Apple Cake           |10           |4.8      |98         |
|2022|Teriyaki Chicken Thighs     |10           |4.8      |108        |
|2022|Twice-Baked Potato Casserole|13           |4.62     |62         |
|2022|Simple Au Gratin Potatoes   |17           |4.59     |64         |
|2022|Ravioli Lasagna             |21           |4.57     |213        |
|2022|Bacon Macaroni Salad        |11           |4.36     |39         |
|2022|Skillet Shepherd’s Pie      |17           |4.06     |83         |
|2022|Enchilada Casser-Ole!       |10           |4.0      |22         |
|2022|Big Soft Ginger Cookies     |13           |4.0      |206        |
+----+----------------------------+-------------+---------+-----

### 6.2. Agregacje Wielowymiarowe: `ROLLUP` i `CUBE`

W systemach OLAP standardowy `GROUP BY` nie wystarcza do generowania podsumowań hierarchicznych.
* **`ROLLUP(d.year, d.quarter)`**: Generuje agregacje hierarchiczne (rok -> kwartał -> suma ogólna).
* **`CUBE(d.year, f.stars)`**: Generuje wszystkie możliwe kombinacje podsumowań (analiza wielowymiarowa).

In [13]:
# Przykład ROLLUP: podsumowanie recenzji hierarchicznie wg roku i kwartału
query_rollup = """
SELECT 
    d.year,
    d.quarter,
    COUNT(f.review_id) AS reviews_count,
    ROUND(AVG(f.stars), 2) AS avg_stars
FROM fact_reviews f
JOIN dim_date d ON f.date_key = d.date_key
WHERE d.year IN (2021, 2022)
GROUP BY ROLLUP(d.year, d.quarter)
ORDER BY d.year NULLS LAST, d.quarter NULLS LAST
"""

spark.sql(query_rollup).show(15)

+----+-------+-------------+---------+
|year|quarter|reviews_count|avg_stars|
+----+-------+-------------+---------+
|2021|      1|          108|     4.25|
|2021|      2|        17218|     4.31|
|2021|      3|          163|     3.77|
|2021|      4|          171|     3.95|
|2021|   NULL|        17660|      4.3|
|2022|      1|          190|     4.25|
|2022|      2|          142|     3.65|
|2022|      3|          153|     3.72|
|2022|      4|           37|     3.68|
|2022|   NULL|          522|     3.89|
|NULL|   NULL|        18182|     4.29|
+----+-------+-------------+---------+



### 6.3. Funkcje Okna (Window Functions) w analityce DWH

Funkcje analityczne pozwalają m.in. na rankingowanie i obliczanie miar względem grupy bez redukowania liczby wierszy.

In [14]:
query_window = """
WITH recipe_stats AS (
    SELECT 
        d.year,
        r.recipe_name,
        ROUND(AVG(f.stars), 2) AS avg_stars,
        COUNT(f.review_id) AS review_count
    FROM fact_reviews f
    JOIN dim_recipe r ON f.recipe_key = r.recipe_key
    JOIN dim_date d ON f.date_key = d.date_key
    GROUP BY d.year, r.recipe_name
    HAVING COUNT(f.review_id) >= 5
),
ranked_recipes AS (
    SELECT 
        year,
        recipe_name,
        avg_stars,
        review_count,
        DENSE_RANK() OVER (PARTITION BY year ORDER BY avg_stars DESC, review_count DESC) AS ranking_w_roku
    FROM recipe_stats
)
SELECT 
    year,
    recipe_name,
    avg_stars,
    review_count,
    ranking_w_roku
FROM ranked_recipes
WHERE ranking_w_roku <= 3
ORDER BY year DESC, ranking_w_roku ASC
"""

spark.sql(query_window).show(20, truncate=False)

+----+--------------------------+---------+------------+--------------+
|year|recipe_name               |avg_stars|review_count|ranking_w_roku|
+----+--------------------------+---------+------------+--------------+
|2022|Chicken and Dumplings     |5.0      |9           |1             |
|2022|Cheeseburger Soup         |5.0      |9           |1             |
|2022|Creamy Macaroni and Cheese|5.0      |6           |2             |
|2022|Chocolate Guinness Cake   |5.0      |5           |3             |
|2021|Creamy White Chili        |4.74     |146         |1             |
|2021|Chicken and Dumplings     |4.72     |259         |2             |
|2021|Buttery Cornbread         |4.68     |96          |3             |
+----+--------------------------+---------+------------+--------------+



## 7. Krok 5: Unikalne Cechy Lakehouse (Delta Lake)

W klasycznym Data Lake modyfikacja pojedynczego wiersza wymagała przepisania całego folderu plików Parquet. W architekturze Lakehouse opartej o **Delta Lake** mamy do dyspozycji:
1. **Transakcje ACID oraz operację `MERGE INTO` (Upsert):** Aktualizacja i wstawianie nowych danych w jednej atomowej transakcji.
2. **Time Travel (Podróż w czasie):** Możliwość odpytania tabeli o stan z konkretnej wersji lub momentu w czasie.

### 7.1. Operacja `MERGE INTO` (Upsert)

Zasymulujmy sytuację, w której moderator zaktualizował ocenę recenzji o identyfikatorze `sp_aUSaElGf_14299_c_2G3aneMRgRMZwXqIHmSdXSG1hEM` ze `stars = 5` na `stars = 4` i podniósł `best_score`.

In [15]:
target_delta_table = DeltaTable.forPath(spark, fact_gold_path)

# Sprawdzenie aktualnego stanu rekordu
spark.sql("""
    SELECT review_id, stars, best_score 
    FROM fact_reviews 
    WHERE review_id = 'sp_aUSaElGf_14299_c_2G3aneMRgRMZwXqIHmSdXSG1hEM'
""").show(truncate=False)

+-----------------------------------------------+-----+----------+
|review_id                                      |stars|best_score|
+-----------------------------------------------+-----+----------+
|sp_aUSaElGf_14299_c_2G3aneMRgRMZwXqIHmSdXSG1hEM|5    |527       |
+-----------------------------------------------+-----+----------+



In [16]:
# Przygotowanie ramki z zaktualizowaną wartością
updates_df = spark.createDataFrame([
    ("sp_aUSaElGf_14299_c_2G3aneMRgRMZwXqIHmSdXSG1hEM", 4, 999)
], ["review_id", "new_stars", "new_best_score"])

# Wykonanie atomowej operacji MERGE
(
    target_delta_table.alias("target")
    .merge(
        updates_df.alias("updates"),
        "target.review_id = updates.review_id"
    )
    .whenMatchedUpdate(set={
        "stars": "updates.new_stars",
        "best_score": "updates.new_best_score"
    })
    .execute()
)

print("Operacja MERGE została pomyślnie zrealizowana!")

Operacja MERGE została pomyślnie zrealizowana!


In [17]:
# Weryfikacja po aktualizacji
spark.read.format("delta").load(fact_gold_path).filter(
    col("review_id") == "sp_aUSaElGf_14299_c_2G3aneMRgRMZwXqIHmSdXSG1hEM"
).select("review_id", "stars", "best_score").show(truncate=False)

+-----------------------------------------------+-----+----------+
|review_id                                      |stars|best_score|
+-----------------------------------------------+-----+----------+
|sp_aUSaElGf_14299_c_2G3aneMRgRMZwXqIHmSdXSG1hEM|4    |999       |
+-----------------------------------------------+-----+----------+



### 7.2. Time Travel (Podróż w czasie)

Delta Lake przechowuje dziennik transakcji w katalogu `_delta_log`. Możemy sprawdzić historię wersji tabeli oraz odczytać stan sprzed operacji `MERGE`!

In [18]:
# Wyświetlenie historii transakcji tabeli fact_reviews
target_delta_table.history().select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

+-------+-----------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation|operationMetrics                                                                                                                                                                                                                             

In [19]:
# Odczyt wersji pierwotnej (versionAsOf = 0) vs wersji bieżącej (versionAsOf = 1)
df_v0 = spark.read.format("delta").option("versionAsOf", 0).load(fact_gold_path)
df_v1 = spark.read.format("delta").option("versionAsOf", 1).load(fact_gold_path)

print("Stan rekordu w wersji 0 (przed MERGE):")
df_v0.filter(col("review_id") == "sp_aUSaElGf_14299_c_2G3aneMRgRMZwXqIHmSdXSG1hEM").select("review_id", "stars", "best_score").show(truncate=False)

print("Stan rekordu w wersji 1 (po MERGE):")
df_v1.filter(col("review_id") == "sp_aUSaElGf_14299_c_2G3aneMRgRMZwXqIHmSdXSG1hEM").select("review_id", "stars", "best_score").show(truncate=False)

Stan rekordu w wersji 0 (przed MERGE):
+-----------------------------------------------+-----+----------+
|review_id                                      |stars|best_score|
+-----------------------------------------------+-----+----------+
|sp_aUSaElGf_14299_c_2G3aneMRgRMZwXqIHmSdXSG1hEM|5    |527       |
+-----------------------------------------------+-----+----------+

Stan rekordu w wersji 1 (po MERGE):
+-----------------------------------------------+-----+----------+
|review_id                                      |stars|best_score|
+-----------------------------------------------+-----+----------+
|sp_aUSaElGf_14299_c_2G3aneMRgRMZwXqIHmSdXSG1hEM|4    |999       |
+-----------------------------------------------+-----+----------+



## 8. Zadania

Poniższe zadania sprawdzają zrozumienie koncepcji modelowania wymiarowego, analityki SQL oraz operacji w architekturze Lakehouse.
Do każdego zadania dołączono praktyczne wskazówki oraz odnośniki do dokumentacji funkcji PySpark, Spark SQL oraz Delta Lake.


### Zadanie 1: Wzbogacenie wymiaru użytkowników i analiza segmentowa

Wykorzystując tabelę `dim_user` oraz `fact_reviews`:
1. Dodaj do `dim_user` kolumnę `user_segment` według reguły opartej na reputacji (`user_reputation`):
   * `user_reputation <= 1` $\rightarrow$ `'Nowy'`
   * `user_reputation` pomiędzy 2 a 10 $\rightarrow$ `'Aktywny'`
   * `user_reputation > 10` $\rightarrow$ `'Lider Społeczności'`
2. Zapisz zaktualizowany wymiar do Delty (nadpisując `dim_user`).
3. Napisz zapytanie w Spark SQL, które dla każdego segmentu użytkowników wyświetli:
   * Liczbę unikalnych użytkowników w segmencie,
   * Łączną liczbę wystawionych recenzji,
   * Średnią ocenę (`stars`) wystawianą przez użytkowników danego segmentu (zaokrągloną do 2 miejsc po przecinku).

---

#### Wskazówki i przydatna dokumentacja:
* **Tworzenie kolumny warunkowej:** Wykorzystaj metodę [`DataFrame.withColumn()`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.withColumn.html) oraz kombinację funkcji [`when()`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.when.html) i `otherwise()`. Przy łączeniu warunków logicznych w PySpark pamiętaj o operatorze bitowym `&` i nawiasach wokół poszczególnych warunków, np. `(col(...) >= 2) & (col(...) <= 10)`.
* **Nadpisywanie tabeli Delta ze zmianą schematu:** Zastosuj tryb zapisu `.mode("overwrite")` wraz z opcją `.option("overwriteSchema", "true")`. Szczegóły znajdziesz w dokumentacji [Delta Lake – Overwrite table schema](https://docs.delta.io/latest/delta-batch.html#overwrite-table-schema).
* **Aktualizacja widoku SQL:** Po nadpisaniu tabeli Delta zaktualizuj zarejestrowany widok tymczasowy wywołując [`createOrReplaceTempView("dim_user")`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.createOrReplaceTempView.html), aby nowe zapytania SQL odwoływały się do odświeżonego schematu.
* **Złączenie wymiaru z faktami:** W zapytaniu SQL użyj złączenia [`LEFT JOIN`](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-join.html) (`dim_user` z `fact_reviews` po kluczu `user_key`), aby nie pominąć użytkowników, którzy jeszcze nie wystawili żadnej recenzji.
* **Agregacje SQL:** Użyj funkcji agregujących [`COUNT(DISTINCT ...)`](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-groupby.html) do policzenia unikalnych użytkowników, `COUNT(...)` dla recenzji oraz [`ROUND(AVG(...), 2)`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.round.html) do wyliczenia zaokrąglonej średniej gwiazdek. Pogrupuj wyniki za pomocą `GROUP BY u.user_segment` i posortuj `ORDER BY`.


In [ ]:
# TODO: Miejsce na Twoje rozwiązanie Zadania 1


### Zadanie 2: Analiza zaufania – najbardziej pomocne recenzje (OLAP Spark SQL)

Napisz zapytanie w Spark SQL, które wyznaczy dla każdego roku 2 recenzje o najwyższym wskaźniku "pomocności" (`thumbs_up`).
Zapytanie powinno zwrócić:
* Rok (`year`),
* Nazwę przepisu (`recipe_name`),
* Nazwę użytkownika (`user_name`),
* Ocenę gwiazdkową (`stars`),
* Liczbę polubień (`thumbs_up`),
* Długość recenzji w znakach (`review_char_count`).

Użyj funkcji okna `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)`. Odfiltruj recenzje, które mają `thumbs_up = 0`.

---

#### Wskazówki i przydatna dokumentacja:
* **Złączenie Schematu Gwiazdy (Star Join):** Połącz centralną tabelę faktów `fact_reviews` z odpowiednimi wymiarami za pomocą [`JOIN`](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-join.html):
  * `dim_date` (`f.date_key = d.date_key`) – aby uzyskać kolumnę `year`,
  * `dim_recipe` (`f.recipe_key = r.recipe_key`) – aby pobrać `recipe_name`,
  * `dim_user` (`f.user_key = u.user_key`) – aby pobrać `user_name`.
* **Funkcja okna (Window Function):** Zastosuj funkcję analityczną [`ROW_NUMBER()`](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-window.html) z partycjonowaniem po roku i sortowaniem malejącym po liczbie polubień:  
  `ROW_NUMBER() OVER (PARTITION BY d.year ORDER BY f.thumbs_up DESC, f.best_score DESC) AS rnk`. Jako kryterium rozstrzygania remisów (*tie-breaker*) możesz dodać drugie pole sortowania (np. `best_score DESC`). Więcej informacji: [Spark SQL Window Syntax](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-window.html).
* **Filtrowanie wyników okna poprzez CTE:** Funkcje okna nie mogą być bezpośrednio użyte w klauzuli `WHERE` tego samego poziomu zapytania. Zdefiniuj zapytanie bazowe w ramach CTE za pomocą klauzuli [`WITH <nazwa_cte> AS (...)`](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-cte.html), a w nadrzędnym zapytaniu odfiltruj najlepsze recenzje warunkiem `WHERE rnk <= 2`.
* **Warunek filtrowania polubień:** Odfiltrowanie recenzji bez polubień (`f.thumbs_up > 0`) umieść w klauzuli `WHERE` zapytania wewnętrznego (CTE), zanim zostanie wyliczony ranking.
* **Sortowanie końcowe:** Uporządkuj finalny wynik za pomocą `ORDER BY year DESC, thumbs_up DESC`.


In [ ]:
# TODO: Miejsce na Twoje rozwiązanie Zadania 2


### Zadanie 3: Wzbogacenie tabeli faktów o nową miarę i audyt w Time Travel

W nowoczesnych hurtowniach danych format Delta Lake pozwala na bezpieczną ewolucję schematu (*Schema Evolution*):
1. Odczytaj tabelę faktów `fact_reviews`.
2. Dodaj nową kolumnę logiczną `is_high_score`: wartość `True`, jeżeli `best_score >= 500`, w przeciwnym razie `False`.
3. Zapisz tabelę do tej samej ścieżki Delty (`fact_gold_path`) z opcją `.option("mergeSchema", "true")`.
4. Wykorzystując funkcję `history()` oraz odczyt z `versionAsOf`, porównaj liczbę kolumn w tabeli faktów w wersji poprzedniej oraz w najnowszej wersji.

---

#### Wskazówki i przydatna dokumentacja:
* **Odczyt tabeli Delta:** Załaduj aktualną tabelę za pomocą [`spark.read.format("delta").load(fact_gold_path)`](https://docs.delta.io/latest/delta-batch.html#read-a-table).
* **Dodanie nowej kolumny logicznej:** Użyj metody [`withColumn()`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.withColumn.html) oraz funkcji [`when(col("best_score") >= 500, True).otherwise(False)`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.when.html).
* **Ewolucja schematu (*Schema Evolution*):** Zapisz zmodyfikowaną ramkę danych z powrotem do `fact_gold_path` z trybem `.mode("overwrite")` i kluczową flagą `.option("mergeSchema", "true")`. Flaga ta instruuje silnik Delty, by zaktualizował schemat tabeli w dzienniku transakcji bez odrzucania operacji. Zobacz: [Delta Lake – Automatic schema evolution](https://docs.delta.io/latest/delta-batch.html#automatic-schema-evolution).
* **Inspekcja historii tabeli:** Użyj klasy `DeltaTable`:  
  [`fact_table = DeltaTable.forPath(spark, fact_gold_path)`](https://docs.delta.io/latest/api/python/spark/index.html#delta.tables.DeltaTable.forPath), a następnie wywołaj [`fact_table.history()`](https://docs.delta.io/latest/api/python/spark/index.html#delta.tables.DeltaTable.history), aby uzyskać historię transakcji z `_delta_log`. Najnowszy numer wersji możesz odczytać np. przez `.select("version").first()["version"]`.
* **Podróż w czasie (*Time Travel*):** Wczytaj poprzednią wersję (`version - 1`) oraz wersję najnowszą, używając opcji [`versionAsOf`](https://docs.delta.io/latest/delta-batch.html#query-an-older-snapshot-of-a-table-time-travel):  
  `spark.read.format("delta").option("versionAsOf", wersja).load(fact_gold_path)`.
* **Weryfikacja kolumn:** Porównaj listę kolumn za pomocą atrybutu [`DataFrame.columns`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.columns.html) oraz ich liczbę za pomocą `len(df.columns)`.


In [ ]:
# TODO: Miejsce na Twoje rozwiązanie Zadania 3


## 9. Zatrzymanie sesji Spark

Po zakończeniu pracy zwalniamy zasoby lokalnego klastra Spark.

In [ ]:
spark.stop()
print("Sesja SparkContext została pomyślnie zamknięta.")